# Delta Lake Basics — 05: Time Travel

## What you will learn
Time travel is one of Delta's most powerful features — query your data
as it looked at any point in the past.

In this notebook:
1. How time travel works — snapshots in `_delta_log`
2. `VERSION AS OF` — query by version number
3. `TIMESTAMP AS OF` — query by point in time
4. `RESTORE` — rolling back a table to a previous state
5. Practical use cases — debugging, auditing, ML reproducibility
6. Retention — how long versions are available


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

In [ ]:
TABLE = f"{DATA_DIR}/05_time_travel"
shutil.rmtree(TABLE, ignore_errors=True)

# Create version history
df.write.format("delta").mode("overwrite").save(TABLE)
print("Version 0: initial insert")

df.limit(5000).write.format("delta").mode("append").save(TABLE)
print("Version 1: appended 5000 rows")

spark.sql(f"UPDATE delta.`{TABLE}` SET status='shipped' WHERE status='confirmed' AND order_id % 2 = 0")
print("Version 2: updated confirmed → shipped")

spark.sql(f"DELETE FROM delta.`{TABLE}` WHERE status='cancelled' AND price < 50")
print("Version 3: deleted cheap cancelled orders")

# Show history
print()
DeltaTable.forPath(spark, TABLE).history() \
    .select("version","timestamp","operation","operationMetrics") \
    .orderBy("version").show(truncate=False)

## Step 1 — VERSION AS OF


In [ ]:
# Query each version
for v in range(4):
    count = spark.read.format("delta") \
                 .option("versionAsOf", v) \
                 .load(TABLE).count()
    print(f"Version {v}: {count:,} rows")

print()
# Compare data between versions
v0_elec = spark.read.format("delta").option("versionAsOf",0).load(TABLE) \
               .filter(F.col("category")=="Electronics").count()
v3_elec = spark.read.format("delta").load(TABLE) \
               .filter(F.col("category")=="Electronics").count()
print(f"Electronics count: V0={v0_elec:,}  →  V3={v3_elec:,}")

## Step 2 — TIMESTAMP AS OF


In [ ]:
import datetime as dt

# Get timestamps from history
history = DeltaTable.forPath(spark, TABLE).history() \
                    .select("version","timestamp").orderBy("version").collect()

print("Timestamp-based time travel:")
for row in history:
    # Query as of 1 second after each commit
    ts = row["timestamp"]
    if ts:
        ts_str = ts.strftime("%Y-%m-%d %H:%M:%S")
        try:
            count = spark.read.format("delta") \
                         .option("timestampAsOf", ts_str) \
                         .load(TABLE).count()
            print(f"  As of {ts_str}: {count:,} rows  (version {row['version']})")
        except Exception as e:
            print(f"  As of {ts_str}: {type(e).__name__}")

## Step 3 — RESTORE: Rolling Back

RESTORE creates a NEW version that points to old data.
The old history is preserved — you can always undo a RESTORE.


In [ ]:
count_v3 = spark.read.format("delta").load(TABLE).count()
print(f"Current (v3): {count_v3:,} rows")

# Restore to version 0
spark.sql(f"RESTORE TABLE delta.`{TABLE}` TO VERSION AS OF 0")

count_after_restore = spark.read.format("delta").load(TABLE).count()
print(f"After RESTORE to v0: {count_after_restore:,} rows")
print()
print("History after RESTORE:")
DeltaTable.forPath(spark, TABLE).history() \
    .select("version","operation").orderBy("version").show()
print("RESTORE creates a new version — history is never deleted")

## Step 4 — Practical Use Cases


In [ ]:
# Use case 1: Debug — "what changed between yesterday and today?"
v_before = 1  # yesterday's version
v_after  = 3  # today's version

before_df = spark.read.format("delta").option("versionAsOf", v_before).load(TABLE)
after_df  = spark.read.format("delta").load(TABLE)

# Find deleted rows
deleted = before_df.join(after_df.select("order_id"), "order_id", "left_anti")
print(f"Use case 1 — Rows deleted between v{v_before} and v{v_after}: {deleted.count():,}")
deleted.show(5)

# Use case 2: ML reproducibility — train model on version 0 data
print("\nUse case 2 — ML training always on version 0 (reproducible):")
train_df = spark.read.format("delta").option("versionAsOf", 0).load(TABLE)
print(f"  Training set: {train_df.count():,} rows (stable, always the same)")

# Use case 3: Audit — who changed order 42?
print("\nUse case 3 — Track order 42 across versions:")
for v in range(4):
    try:
        row = spark.read.format("delta").option("versionAsOf", v).load(TABLE) \
                   .filter(F.col("order_id") == 42) \
                   .select("order_id","status","revenue").collect()
        if row:
            print(f"  Version {v}: status={row[0]['status']} revenue={row[0]['revenue']}")
    except Exception:
        pass

## Summary: Time Travel Reference

```python
# By version number
spark.read.format("delta").option("versionAsOf", 5).load(path)

# By timestamp
spark.read.format("delta").option("timestampAsOf", "2024-01-15 10:00:00").load(path)

# SQL syntax
spark.sql("SELECT * FROM delta.`/path` VERSION AS OF 5")
spark.sql("SELECT * FROM delta.`/path` TIMESTAMP AS OF '2024-01-15'")

# RESTORE (creates new version pointing to old state)
spark.sql("RESTORE TABLE delta.`/path` TO VERSION AS OF 5")
spark.sql("RESTORE TABLE delta.`/path` TO TIMESTAMP AS OF '2024-01-15'")
```

### Retention period
- Default: 30 days (controlled by `delta.deletedFileRetentionDuration`)
- VACUUM removes files older than retention → time travel limited to retention window
- Set longer retention: `ALTER TABLE SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 60 days')`
